In [2]:
import os
import glob
import re
import json
import time
import datetime
import xarray as xr
import xcdat as xc
import numpy as np
import pandas as pd
import shutil
import psutil
import subprocess
from pathlib import Path

In [3]:
def read_var_dict():
    var_dict = {
        'iceArea'     : {"unit": "m$^2$",        "min": -6,  "max": 6,  "nlev": 11},  # percentage points
        'iceVolume'   : {"unit": "m$^3$",        "min": -0.5,"max": 0.5,"nlev": 11},  # thickness anomalies usually meters
        'iceThickness': {"unit": "m",            "min": -0.5,"max": 0.5,"nlev": 11},  # thickness anomalies usually meters
        }
    return var_dict 

In [8]:
class EnsembleDataProcs:
    def __init__(
        self, regnam, data_dir, file_template, num_ens, var_dict,
        out_dir=None, chunks=None, fontz=12, keep_vars=None
    ):
        self.regnam     = regnam
        self.data_dir   = data_dir
        self.out_dir    = os.path.join(out_dir, regnam)
        self.file_tmpl  = file_template
        self.num_ens    = num_ens
        self.var_dict   = var_dict
        self.fontz      = fontz
        self.chunks     = chunks
        self.ds_all     = None
        self.time_coord = None
        self.keep_vars  = set(keep_vars or ["area"])

    # --- fix _ensemble_paths (simpler & no walrus confusion) ---
    def _ensemble_paths(self):
        paths = []
        for i in range(0, self.num_ens + 1):
            ens_str = str(i).zfill(2)
            fname = self.file_tmpl.replace("%(ENS)", ens_str)
            fpath = os.path.join(self.data_dir, fname)
            print(fpath)
            if os.path.isfile(fpath):
                paths.append(fpath)
        if not paths:
            raise FileNotFoundError("No ensemble files found after filtering missing files.")
        return sorted(paths)

    def _preprocess(self, ds: xr.Dataset) -> xr.Dataset:
        keep_data = list(ds.data_vars)
        keep_aux  = [v for v in self.keep_vars if v in ds]
        out = ds[keep_data + keep_aux]
        for c in ("year", "time", "lat", "lon"):
            if c in out and c not in out.coords:
                out = out.set_coords(c)
        return out

    def _detect_time_coord(self, ds: xr.Dataset) -> str:
        if "year" in ds.coords or "year" in ds:
            return "year"
        if "time" in ds.coords:
            return "time"
        raise ValueError("No 'year' or 'time' coordinate found.")

    @staticmethod
    def _compressible_encoding(ds, complevel=4):
        enc = {}
        for v in ds.data_vars:
            enc[v] = {"zlib": True, "complevel": complevel, "shuffle": True}
            if np.issubdtype(ds[v].dtype, np.floating) and ds[v].dtype == np.float32:
                enc[v]["dtype"] = "float32"
        return enc

    # ---------- weights ----------
    def _get_weights(self, tmpl_da: xr.DataArray, prefer="area", area_var="area"):
        lat_name = next((c for c in ("lat", "latitude", "y") if c in tmpl_da.coords), None)
        lon_name = next((c for c in ("lon", "longitude", "x") if c in tmpl_da.coords), None)
        if lat_name is None or lon_name is None:
            return None

        ds = tmpl_da.to_dataset(name="_tmp_")
        if prefer == "area" and area_var in ds:
            w = ds[area_var]
        elif prefer == "area" and area_var in ds.coords:
            w = ds.coords[area_var]
        else:
            w = np.cos(np.deg2rad(ds[lat_name]))
            sel = {d: 0 for d in ds["_tmp_"].dims if d not in (lat_name, lon_name)}
            base = ds["_tmp_"].isel(**sel)
            w, _ = xr.broadcast(w, base)
            w = w.squeeze(drop=True)
        w = xr.where(np.isfinite(w) & (w > 0), w, 0)
        return w

    def _weights_for_var(self, da: xr.DataArray, weights="auto", area_var="area"):
        if weights in ("auto", "area", "coslat"):
            prefer = "area" if weights in ("auto", "area") else "coslat"
            w = self._get_weights(
                da.isel(ens=0) if "ens" in da.dims else da,
                prefer=prefer, area_var=area_var
            )
            if weights == "area" and w is None:
                raise ValueError("Requested 'area' weights but no area/lat-lon found.")
            return w
        elif weights is None:
            return None
        else:
            raise ValueError("weights must be 'auto', 'area', 'coslat', or None")

    def _detect_time_in_file(self, ds: xr.Dataset):
        """Return ('year'|'time', values, 'numpy'|'cftime')."""
        if "year" in ds.dims:
            return "year", ds["year"].values, "numeric"
        if "time" in ds.dims:
            tvals = ds["time"].values
            kind = "numpy" if np.issubdtype(tvals.dtype, np.datetime64) else "cftime"
            return "time", tvals, kind
        raise ValueError("Target file has neither 'year' nor 'time' dimension.")

    def _coerce_out_to_target_time(
        self, out_ds: xr.Dataset, tgt_dim: str, tgt_kind: str, tgt_time_var: xr.DataArray = None
    ):
        """
        Make out_ds use the same time dim/type as the target file.
        If tgt_dim=='time' and target uses a CF calendar, create Jan-01 for each year in that calendar.
        """
        if "year" not in out_ds.dims and "time" not in out_ds.dims:
            raise ValueError("Outgoing dataset has no 'year' or 'time' dimension.")

        if tgt_dim == "year":
            if "year" in out_ds.dims:
                return out_ds
            if "time" in out_ds.dims:
                years = out_ds["time"].dt.year
                return (out_ds
                        .assign_coords(year=("time", years.values))
                        .swap_dims({"time": "year"})
                        .drop_vars("time"))
            return out_ds

        if tgt_dim == "time":
            if "time" in out_ds.dims:
                return out_ds
            years = out_ds["year"].values
            if tgt_kind == "numpy":
                tvals = np.array([np.datetime64(f"{int(y):04d}-01-01") for y in years])
            else:
                import cftime
                cal = None
                if tgt_time_var is not None:
                    cal = tgt_time_var.encoding.get("calendar") or tgt_time_var.attrs.get("calendar")
                cal = (cal or "gregorian").lower()
                cal_map = {
                    "gregorian": cftime.DatetimeGregorian,
                    "standard":  cftime.DatetimeGregorian,
                    "proleptic_gregorian": cftime.DatetimeProlepticGregorian,
                    "noleap":    cftime.DatetimeNoLeap,
                    "365_day":   cftime.DatetimeNoLeap,
                    "all_leap":  cftime.DatetimeAllLeap,
                    "366_day":   cftime.DatetimeAllLeap,
                    "360_day":   cftime.Datetime360Day,
                }
                cls = cal_map.get(cal, cftime.DatetimeGregorian)
                tvals = np.array([cls(int(y), 1, 1) for y in years], dtype=object)
            return (out_ds
                    .assign_coords(time=("year", tvals))
                    .swap_dims({"year": "time"})
                    .drop_vars("year"))

        raise ValueError(f"Unhandled target time dim: {tgt_dim}")

    def _ensure_same_time_axis(self, out_ds: xr.Dataset, out_path: str, dim: str) -> xr.Dataset:
        """Ensure out_ds[dim] matches the target file's coord exactly.
           If values match but order differs, reorder to target. Else raise."""
        with xr.open_dataset(out_path, decode_cf=True) as tgt:
            if dim not in tgt.coords:
                raise ValueError(f"Target file lacks '{dim}' coordinate.")
            tgt_coord = tgt[dim]
            src_coord = out_ds[dim]

            # Type/calendar compatibility
            tgt_is_np = np.issubdtype(tgt_coord.values.dtype, np.datetime64)
            src_is_np = np.issubdtype(src_coord.values.dtype, np.datetime64)
            if tgt_is_np != src_is_np:
                raise ValueError(f"Time type mismatch: target uses "
                                 f"{'numpy datetime64' if tgt_is_np else 'cftime'}, "
                                 f"but outgoing uses {'numpy datetime64' if src_is_np else 'cftime'}.")

            # Calendar check for cftime
            if not tgt_is_np:
                cal_tgt = (tgt_coord.encoding.get("calendar")
                           or tgt_coord.attrs.get("calendar") or "gregorian").lower()
                cal_src = (src_coord.encoding.get("calendar")
                           or src_coord.attrs.get("calendar") or "gregorian").lower()
                if cal_tgt != cal_src:
                    raise ValueError(f"Calendar mismatch: target '{cal_tgt}' vs outgoing '{cal_src}'.")

            # Length must match
            if src_coord.size != tgt_coord.size:
                raise ValueError(f"Time length mismatch: target has {tgt_coord.size}, "
                                 f"outgoing has {src_coord.size}.")

            # Exact equality (including order)
            same_order = np.array_equal(src_coord.values, tgt_coord.values)
            if same_order:
                return out_ds  # perfect match

            # If values match as a set, reorder; else error
            if np.array_equal(np.sort(src_coord.values), np.sort(tgt_coord.values)):
                return out_ds.reindex({dim: tgt_coord.values}, copy=False)

            raise ValueError("Time values differ between outgoing and target; "
                             "this path only appends variables to an existing, identical time axis.")

    def save_monthly_timeseries_split(
        self,
        out_dir=None,
        output_template=None,
        time_slice=None,         # e.g., ("1850-01-01","2050-12-31") or slice(...)
        compute=True,
        complevel=4,
        overwrite=False,
        # --- anomaly options ---
        anomaly=None,            # None | "year" | "clim"
        ref_year=None,           # used when anomaly == "year"
        ref_period=None,         # (start, end) used when anomaly == "clim", e.g. ("1981-01-01","2010-12-31")
    ):
        """
        Save MONTHLY base data (per input ensemble file) with optional anomalies.

        - No spatial averaging: variables with lat/lon are rejected.
        - Time kept at (monthly) 'time' resolution. If input is higher than monthly,
          it is downsampled to monthly means using resample('MS').
        - If anomaly == "year": subtract that year's monthly cycle (requires 12 months present).
        - If anomaly == "clim": subtract monthly climatology over ref_period (inclusive).

        output_template tokens:
          {base} -> input basename without .nc
          {reg}  -> region name
          %(ENS) -> two-digit ensemble index (if parsed from ".enXX" in filename)
        """
        if output_template is None:
            raise ValueError("Must set 'output_template' for split saves.")

        if anomaly not in (None, "year", "clim"):
            raise ValueError("anomaly must be None, 'year', or 'clim'")

        if anomaly == "year" and ref_year is None:
            raise ValueError("ref_year is required when anomaly == 'year'.")

        if anomaly == "clim" and (not isinstance(ref_period, (list, tuple)) or len(ref_period) != 2):
            raise ValueError("ref_period=('YYYY-MM-DD','YYYY-MM-DD') is required when anomaly == 'clim'.")

        out_root = out_dir or self.out_dir
        Path(out_root).mkdir(parents=True, exist_ok=True)

        for in_path in self._ensemble_paths():
            ds = xr.open_dataset(in_path, chunks=self.chunks)
            ds = self._preprocess(ds)

            # Ensure we have a 'time' dimension for monthly data
            if "time" not in ds.dims:
                raise ValueError(f"{os.path.basename(in_path)} has no 'time' dimension; monthly base requires 'time'.")

            # Optional slice on time
            if time_slice is not None:
                if isinstance(time_slice, slice):
                    ds = ds.sel(time=time_slice)
                elif isinstance(time_slice, (list, tuple)) and len(time_slice) == 2:
                    ds = ds.sel(time=slice(time_slice[0], time_slice[1]))
                else:
                    raise ValueError("time_slice must be a slice or (start, end)")

            # Downsample to monthly means if needed (handles sub-monthly)
            # NOTE: Works with numpy datetime64 and cftime (xarray supports resample with cftime)
            # If already monthly, this is a no-op (means within each month).
            try:
                ds_m = ds.resample(time="MS").mean()
            except Exception:
                # Fallback: groupby year,month to construct monthly—more robust for odd calendars
                yrs = xr.DataArray(ds["time"].dt.year,  dims=("time",), name="year")
                mns = xr.DataArray(ds["time"].dt.month, dims=("time",), name="month")
                ds_m = ds.groupby((yrs, mns)).mean()
                # rebuild a monthly time coord at first day-of-month w/ same calendar if possible
                if np.issubdtype(ds["time"].values.dtype, np.datetime64):
                    tvals = []
                    for y, m in zip(ds_m["year"].values, ds_m["month"].values):
                        y, m = int(y), int(m)
                        tvals.append(np.datetime64(f"{y:04d}-{m:02d}-01"))
                    ds_m = ds_m.rename({"year": "Y", "month": "M"})
                    ds_m = ds_m.assign_coords(time=("Y_M", np.array(tvals)))
                    ds_m = ds_m.swap_dims({"Y_M": "time"}).drop_vars(["Y","M"])
                else:
                    import cftime
                    cal = (ds["time"].encoding.get("calendar")
                           or ds["time"].attrs.get("calendar")
                           or "gregorian").lower()
                    cal_map = {
                        "gregorian": cftime.DatetimeGregorian,
                        "standard":  cftime.DatetimeGregorian,
                        "proleptic_gregorian": cftime.DatetimeProlepticGregorian,
                        "noleap":    cftime.DatetimeNoLeap,
                        "365_day":   cftime.DatetimeNoLeap,
                        "all_leap":  cftime.DatetimeAllLeap,
                        "366_day":   cftime.DatetimeAllLeap,
                        "360_day":   cftime.Datetime360Day,
                    }
                    cls = cal_map.get(cal, cftime.DatetimeGregorian)
                    tvals = []
                    for y, m in zip(ds_m["year"].values, ds_m["month"].values):
                        tvals.append(cls(int(y), int(m), 1))
                    ds_m = ds_m.rename({"year": "Y", "month": "M"})
                    ds_m = ds_m.assign_coords(time=("Y_M", np.array(tvals, dtype=object)))
                    ds_m = ds_m.swap_dims({"Y_M": "time"}).drop_vars(["Y","M"])

            out_vars = {}

            for v in [vv for vv in ds_m.data_vars if vv not in self.keep_vars]:
                da = ds_m[v]

                # Reject spatial (lat/lon) variables: monthly base here is for 1D series
                if any(d in da.dims for d in ("lat", "lon")):
                    raise ValueError(
                        f"Variable '{v}' has spatial dims; monthly base saver expects time series only."
                    )

                base_out = da
                if np.issubdtype(base_out.dtype, np.floating):
                    base_out = base_out.astype("float32")

                out_vars[f"{v}_imsk"] = base_out
                out_vars[f"{v}_imsk"].attrs = ds[v].attrs

                # Optional anomalies
                if anomaly == "year":
                    # select the 12-month slice for that year and subtract month-wise
                    yr_sel = base_out.sel(time=str(ref_year))
                    # Build monthly means for that specific year (should be one value per month)
                    ref_cycle = yr_sel.groupby("time.month").mean("time")
                    anom = base_out.groupby("time.month") - ref_cycle
                    if np.issubdtype(anom.dtype, np.floating):
                        anom = anom.astype("float32")
                    out_vars[f"D{v}_imsk"] = anom
                    out_vars[f"D{v}_imsk"].attrs = {
                        **ds[v].attrs,
                        "description": f"Monthly anomaly relative to {ref_year} monthly cycle",
                    }

                elif anomaly == "clim":
                    start, end = ref_period
                    clim_sel = base_out.sel(time=slice(start, end))
                    if clim_sel.time.size == 0:
                        raise ValueError(f"No data in ref_period {ref_period} for variable '{v}'.")
                    ref_clim = clim_sel.groupby("time.month").mean("time")
                    anom = base_out.groupby("time.month") - ref_clim
                    if np.issubdtype(anom.dtype, np.floating):
                        anom = anom.astype("float32")
                    out_vars[f"D{v}_imsk"] = anom
                    out_vars[f"D{v}_imsk"].attrs = {
                        **ds[v].attrs,
                        "description": f"Monthly anomaly relative to climatology over {start}–{end}",
                    }

            if not out_vars:
                print(f"[WARNING] No variables to write for {os.path.basename(in_path)}; skipping.")
                continue

            out_ds = xr.Dataset(out_vars)

            # mark unlimited dimension
            if "time" not in out_ds.dims:
                raise ValueError("Internal error: expected 'time' dimension after monthly processing.")
            out_ds["time"].encoding["unlimited"] = True

            # Build output path
            base = os.path.basename(in_path)
            if base.endswith(".nc"):
                base = base[:-3]
            ens_str = None
            parts = base.split(".en")
            if len(parts) > 1:
                tail = parts[1]
                ens_str = "".join(ch for ch in tail if ch.isdigit())[:2] or None

            out_fname = output_template.replace("{base}", base).replace("{reg}", str(self.regnam))
            if "%(ENS)" in out_fname and ens_str:
                out_fname = out_fname.replace("%(ENS)", ens_str)
            elif "%(ENS)" in out_fname:
                out_fname = out_fname.replace(".%(ENS)", "").replace("%(ENS)", "")

            out_path = os.path.join(out_root, out_fname)

            if os.path.exists(out_path) and not overwrite:
                print(f"[INFO] Exists, skip (use overwrite=True): {os.path.basename(out_path)}")
                continue

            # Encoding & compression
            enc = self._compressible_encoding(out_ds, complevel=complevel)
            enc["time"] = {"zlib": True, "complevel": complevel, "shuffle": True}

            out_ds.to_netcdf(
                out_path,
                mode="w",
                unlimited_dims=["time"],
                compute=compute,
                encoding=enc,
            )
            print(f"[INFO] Wrote monthly base: {out_path}")


In [10]:
if __name__ == "__main__":
    # --- Paths
    top_dir  = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble"
    data_dir = f"{top_dir}/data/seaice"
    out_dir  = f"{top_dir}/data/global"
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    # --- Variables (not used by monthly saver, but fine to keep)
    var_dict = read_var_dict()

    # --- Ensemble naming
    regnam    = "Arctic"
    group     = "v3.LR.historical"
    nens      = 25
    period    = "1850-2024"   # will become YYYY01-YYYY12
    ref_year  = 1850          # only needed if you compute anomalies later

    # Build YYYY01-YYYY12 for filenames
    start_year, end_year = period.split("-")
    date_span = f"{start_year}01-{end_year}12"

    # File template (expects .en01, .en02, ...)
    file_template = f"{group}.en%(ENS).mpassi_ts.{date_span}.nc"

    results_dir = out_dir
    Path(results_dir).mkdir(parents=True, exist_ok=True)

    # --- Instantiate processor
    chunks = {"ens": 1, "year": 50, "lat": 90, "lon": 180}  # safe to keep; ignored if dims absent
    proc = EnsembleDataProcs(
        regnam=regnam,
        data_dir=data_dir,
        file_template=file_template,
        num_ens=nens,
        var_dict=var_dict,
        out_dir=results_dir,
        chunks=chunks,
        keep_vars=["Time","time","time_bnds"],  # keep time metadata if present
    )

    # Optional time narrowing on monthly data
    time_slice = None  # e.g., ("1981-01-01", "2010-12-31")

    # Output name: this is monthly base (not area mean), so label accordingly
    output_template = f"{group}.en%(ENS).sea_ice.{date_span}.nc"

    # --- Write one monthly-base file per ensemble member (no anomalies)
    proc.save_monthly_timeseries_split(
        out_dir=results_dir,
        output_template=output_template,
        time_slice=time_slice,
        compute=True,
        complevel=4,
        overwrite=False,      # set True to regenerate
        anomaly=None,         # options: None | "year" | "clim"
        # ref_year=ref_year,  # only needed if anomaly="year"
        # ref_period=("1981-01-01","2010-12-31"),  # only if anomaly="clim"
    )

    print("[DONE]")


/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble/data/seaice/v3.LR.historical.en00.mpassi_ts.185001-202412.nc
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble/data/seaice/v3.LR.historical.en01.mpassi_ts.185001-202412.nc
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble/data/seaice/v3.LR.historical.en02.mpassi_ts.185001-202412.nc
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble/data/seaice/v3.LR.historical.en03.mpassi_ts.185001-202412.nc
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble/data/seaice/v3.LR.historical.en04.mpassi_ts.185001-202412.nc
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble/data/seaice/v3.LR.historical.en05.mpassi_ts.185001-202412.nc
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble/data/seaice/v3.LR.historical.en06.mpassi_ts.185001-202412.nc
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble/data/seaice/v3.LR.historical.